# Face recognition

In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals
from tensorflow.python.keras.layers import Dense,Conv2D,Flatten,MaxPooling2D,ZeroPadding2D,Dropout,Softmax
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.python.keras import Sequential
from sklearn.datasets import fetch_lfw_people
from sklearn.datasets import fetch_lfw_pairs
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
#!pip install scikit-plot
import scikitplot as skplt
import numpy as np
import random
import pandas as pd

# Commented out IPython magic to ensure Python compatibility.
import tensorflow_datasets as tfds
import tensorflow as tf

## Prepare Training

El dataset contiene imágenes de 19 personas, no obstante para cada una de ellas hay un número de imágenes diferente. Se escogen solo las personas que tengan 100 imágenes o más.

In [ ]:
# For GOOGLE COLAB
# ==========
# from google.colab import drive
# drive.mount("/content/drive", force_remount=True)
# data_dir = "/content/drive/MyDrive/Colab Notebooks/scikit_learn_data"

In [3]:
# For OS
# ==========
import os
d = os.path.dirname(os.getcwd())
data_zip_dir = os.path.join(d, "data/model_2/scikit_learn_data.zip")

import zipfile
with zipfile.ZipFile(data_zip_dir, 'r') as zip_ref:
    zip_ref.extractall(os.path.join(d, "data/model_2"))

data_dir = os.path.join(d, "data/model_2/scikit_learn_data")

data = fetch_lfw_people(data_home = data_dir, min_faces_per_person=100, resize=1.0, slice_=(slice(60, 188), slice(60, 188)), color=True)

class_count = len(data.target_names)

/Users/sergiocuencanunez/Documents/UIUC/4º GITT+BA/Asignaturas/Anual/TFG | Trabajo de Fin de Grado/Proyecto/TFG.-Reconocimiento-Facial/data/model_2/test


NameError: name 'fetch_lfw_people' is not defined

In [ ]:
from collections import Counter
counts = Counter(data.target)
names = {}

for key in counts.keys():
    names[data.target_names[key]] = counts[key]

df = pd.DataFrame.from_dict(names, orient='index')
df.plot(kind='bar')

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

fig, ax = plt.subplots(3, 6, figsize=(18, 10))

for i, axi in enumerate(ax.flat):
    axi.imshow(data.images[i] / 255) # Scale pixel values so Matplotlib doesn't clip everything above 1.0
    axi.set(xticks=[], yticks=[], xlabel=data.target_names[data.target[i]])

## Balancing the data

Los modelos de clasificación se entrenan mejor con datasets que estén balanceados. Es por ello que se utilizan 100 imágenes para cada persona.

In [ ]:
mask = np.zeros(data.target.shape, dtype=np.bool)

for target in np.unique(data.target):
    mask[np.where(data.target == target)[0][:100]] = 1

x_faces = data.data[mask]
y_faces = data.target[mask]
x_faces = np.reshape(x_faces, (x_faces.shape[0], data.images.shape[1], data.images.shape[2], data.images.shape[3]))
x_faces.shape

In [ ]:
####### Train and test splitting of data #######

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.model_selection import train_test_split

face_images = preprocess_input(np.array(x_faces))
face_labels = to_categorical(y_faces)
x_train, x_test, y_train, y_test = train_test_split(face_images, face_labels, train_size=0.8, stratify=face_labels, random_state=0)

## Use transfer learning with ResNet50 and ImageNet weights

El primer modelo emplea transfer learning con ResNet50 con pesos calculados de una red entranada con más de 1 millón de imágenes del dataset de ImageNet. Se empieza por cargar ResNet50 sin las capas de clasificación y congelando las capas del cuello de botella

In [ ]:
from tensorflow.keras.applications import ResNet50

base_model = ResNet50(weights='imagenet', include_top=False)
base_model.trainable = False

Se añaden las capas de clasificación al modelo de base y se incluye una capa de reajuste para redimensionar las imagenes puestas como entrada a la red a 224x224

In [ ]:
from keras.models import Sequential
from keras.layers import Flatten, Dense, Resizing

model = Sequential()
model.add(Resizing(224, 224))
model.add(base_model)
model.add(Flatten())
model.add(Dense(1024, activation='relu'))
model.add(Dense(class_count, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
import time

start_time = time.time()
hist = model.fit(x_train, y_train, validation_data=(x_test, y_test), batch_size=10, epochs=10)
end_time = time.time()

print("###### Total Time Taken: ", round((end_time - start_time)/60), 'Minutes ######')

In [ ]:
acc = hist.history['accuracy']
val_acc = hist.history['val_accuracy']
epochs = range(1, len(acc) + 1)
plt.plot(epochs, acc, '-', label='Training Accuracy')
plt.plot(epochs, val_acc, ':', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.plot()

## Use transfer learning with ResNet50 and VGGFace2 weights

Inicializada con pesos de ImageNet, ResNet50 logra una precisión de alrededor del 90 % en el reconocimiento  en el conjunto de datos de LFW. Pero los pesos de ImageNet se generaron entrenando la red con imágenes de cientos de objetos diferentes. Estos pesos no están optimizados para extraer características de las caras.

keras-vggface es un paquete de Python que modela CNN preentrenadas para reconocer las caras de personas famosas. Estas CNN (y sus pesos) están encapsuladas en una clase denominada VGGFace. Se cargan las capas de cuello de botella de la versión ResNet50 de VGGFace junto con los pesos a los que se llegó cuando se entrenó con más de 3 millones de imágenes en el conjunto de datos VGGFace2 y se usa el transfer learning para entrenar la red resultante en el conjunto de datos LFW. Como antes, se comienza por cargar VGGFace sin las capas de clasificación y congelando las capas de cuello de botella.

Una capa de cuello de botella, en inglés bottleneck layer, es una capa que contiene pocos nodos en comparación con las capas anteriores. Se puede utilizar para obtener una representación de la entrada con dimensionalidad reducida. Un ejemplo de esto es el uso de codificadores automáticos con capas de cuello de botella para la reducción de dimensionalidad no lineal.
Con ello se reduce el número de parámetros y las multiplicaciones de matrices. La idea es hacer que los bloques residuales sean lo más pequeños posible para aumentar la profundidad y tener menos parámetros.

In [ ]:
import site; print(site.getsitepackages())

In [ ]:
# !pip install git+https://github.com/rcmalli/keras-vggface.git
# !pip install keras_applications --no-deps

In [ ]:
# For WINDOWS
# ==========
# filename = "C:/Users/lfsanchez/Anaconda3/Lib/site-packages/keras_vggface/models.py"
# f=open(filename)
# text = f.read()
# f.close()
# f=open(filename, "w+")
# f.write(text.replace('keras.engine.topology', 'tensorflow.keras.utils'))
# f.close()

In [ ]:
# For UNIX
# ==========
# filename = "/usr/local/lib/python3.8/dist-packages/keras_vggface/models.py"
# text = open(filename).read()
# open(filename, "w+").write(text.replace('keras.engine.topology', 'tensorflow.keras.utils'))
import tensorflow as tf

from keras_vggface.vggface import VGGFace

base_model = VGGFace(model='resnet50', include_top=False)
base_model.trainable = False

Se añaden las capas de clasificación al modelo de base y se incluye una capa de reajuste para redimensionar las imagenes puestas como entrada a la red a 224x224

In [ ]:
from keras.models import Sequential
from keras.layers import Flatten, Dense, Resizing

model = Sequential()
model.add(Resizing(224, 224))
model.add(base_model)
model.add(Flatten())
model.add(Dense(1024, activation='relu'))
model.add(Dense(class_count, activation='softmax')) # Terminar con una capa que contenga el número de clases que puede predecir el modelo
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

La función relu ayuda a prevenir el crecimiento exponencial en la computación requerida para operar la red neuronal. Si la CNN aumenta en tamaño, el coste computacional de agregar ReLU adicionales aumenta linealmente.
La función softmax se utiliza como función de activación en la capa de salida de los modelos de redes neuronales que predicen una distribución de probabilidad multinomial

In [ ]:
import time

start_time = time.time()
hist = model.fit(x_train, y_train, validation_data=(x_test, y_test), batch_size=10, epochs=10)
end_time = time.time()

print("###### Total Time Taken: ", round((end_time - start_time)/60), 'Minutes ######')

In [ ]:
acc = hist.history['accuracy']
val_acc = hist.history['val_accuracy']
epochs = range(1, len(acc) + 1)
plt.plot(epochs, acc, '-', label='Training Accuracy')
plt.plot(epochs, val_acc, ':', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.plot()

In [ ]:
score = model.evaluate(x_train,y_train,verbose=0)
print(score[1]*100)